# Extract rectum patches by gender and split


In [ ]:
import os
import SimpleITK as sitk
import numpy as np

DATA_SPLIT = "train_val"  # options: "train_val", "test"
GENDER = "male"  # options: "male", "female"

BASE_DIR = r"/path/to/workspace/classification_model_originalimage/totalsegment_work/resampled"
input_dir = os.path.join(BASE_DIR, f"{GENDER}_{DATA_SPLIT}")
output_dir = os.path.join(BASE_DIR, f"{GENDER}_{DATA_SPLIT}_patches")
os.makedirs(output_dir, exist_ok=True)


In [ ]:
def bounding_box_male(prostate_img, colon_img):
    prostate_array = sitk.GetArrayFromImage(prostate_img)
    colon_array = sitk.GetArrayFromImage(colon_img)
    prostate_indices = np.nonzero(prostate_array)
    y_min = np.min(prostate_indices[1])
    rectum_array = colon_array.copy()
    rectum_array[:, y_min + 5 :, :] = 0
    return _expanded_box(rectum_array, prostate_img.GetSize(), expansion=40)


def bounding_box_female(colon_mask):
    colon_array = sitk.GetArrayFromImage(colon_mask)
    colon_indices = np.nonzero(colon_array)
    min_y = np.min(colon_indices[1])
    voxel_height = colon_mask.GetSpacing()[1]
    num_voxels = int(40 / voxel_height)
    rectum_max_y = min(min_y + num_voxels, colon_array.shape[1] - 1)
    rectum_array = np.zeros_like(colon_array)
    rectum_array[:, min_y : rectum_max_y + 1, :] = colon_array[:, min_y : rectum_max_y + 1, :]
    return _expanded_box(rectum_array, colon_mask.GetSize(), expansion=20)


def _expanded_box(mask_array, image_size, expansion):
    indices = np.nonzero(mask_array)
    min_z, max_z = np.min(indices[0]), np.max(indices[0])
    min_y, max_y = np.min(indices[1]), np.max(indices[1])
    min_x, max_x = np.min(indices[2]), np.max(indices[2])
    min_z = max(min_z - expansion, 0)
    max_z = min(max_z + expansion, image_size[2] - 1)
    min_y = max(min_y - expansion, 0)
    max_y = min(max_y + expansion, image_size[1] - 1)
    min_x = max(min_x - expansion, 0)
    max_x = min(max_x + expansion, image_size[0] - 1)
    return min_z, max_z, min_y, max_y, min_x, max_x


In [ ]:
def extract_patches(min_z, max_z, min_y, max_y, min_x, max_x, mr_image):
    roi_filter = sitk.RegionOfInterestImageFilter()
    roi_filter.SetSize([int(max_x - min_x + 1), int(max_y - min_y + 1), int(max_z - min_z + 1)])
    roi_filter.SetIndex([int(min_x), int(min_y), int(min_z)])
    return roi_filter.Execute(mr_image)


for foldername in os.listdir(input_dir):
    file_path = os.path.join(input_dir, foldername)
    image = sitk.ReadImage(os.path.join(file_path, "resampled_image.nii.gz"))
    colon_img = sitk.ReadImage(os.path.join(file_path, "colon.nii.gz"))

    try:
        if GENDER == "male":
            prostate_img = sitk.ReadImage(os.path.join(file_path, "prostate.nii.gz"))
            box = bounding_box_male(prostate_img, colon_img)
        else:
            box = bounding_box_female(colon_img)

        rectum_patch = extract_patches(*box, image)
        sitk.WriteImage(rectum_patch, os.path.join(output_dir, foldername + ".nii.gz"))
    except Exception as e:
        print(f"An error occurred: {e}")
        print("patient number:", foldername)

print("done")
